In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
from PIL import Image
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [8]:
class DQN(nn.Module):
    """Deep Q-Network architecture for Atari games"""
    def __init__(self, input_channels, n_actions):
        super(DQN, self).__init__()
        
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        
        self.fc_input_dim = self._get_conv_output_dim(input_channels)
        
        self.fc1 = nn.Linear(self.fc_input_dim, 512)
        self.fc2 = nn.Linear(512, n_actions)

    def _get_conv_output_dim(self, input_channels):
        """Calculate the output dimension of the convolutional layers"""
        with torch.no_grad():
            x = torch.zeros(1, input_channels, 84, 84)
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            return x.view(1, -1).size(1)
    
    def forward(self, x):
        x = x / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [9]:
class ReplayBuffer:
    """Replay buffer to store and sample experiences"""
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (np.stack(state), np.array(action), np.array(reward, dtype=np.float32),
                np.stack(next_state), np.array(done, dtype=np.bool_))
    
    def __len__(self):
        return len(self.buffer)

In [10]:
class PreprocessAtari:
    """Preprocess Atari frames"""
    def __init__(self, height=84, width=84):
        self.height = height
        self.width = width
    
    def __call__(self, frame):
        if len(frame.shape) == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        frame = cv2.resize(frame, (self.width, self.height), interpolation=cv2.INTER_AREA)
        
        frame = frame.astype(np.float32) / 255.0
        
        return frame

In [11]:
class DQNAgent:
    """DQN A                gent"""
    def __init__(self, env_name, lr=0.0001, gamma=0.99, epsilon_start=1.0, 
                 epsilon_end=0.01, epsilon_decay=0.995, buffer_capacity=100000, 
                 batch_size=32, target_update_freq=1000):
        
        self.env = gym.make(env_name, render_mode='rgb_array')
        self.preprocess = PreprocessAtari()
        
        self.n_actions = self.env.action_space.n
        self.input_channels = 4
        
        self.q_network = DQN(self.input_channels, self.n_actions).to(device)
        self.target_network = DQN(self.input_channels, self.n_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        
        self.replay_buffer = ReplayBuffer(buffer_capacity)
        
        self.steps = 0
        self.episode_rewards = []
        self.losses = []
        
    def preprocess_state(self, state):
        """Preprocess a single frame"""
        return self.preprocess(state)
    
    def stack_frames(self, frames):
        """Stack multiple frames"""
        return np.stack(frames, axis=0)
    
    def select_action(self, state, training=True):
        """Select action using epsilon-greedy policy"""
        if training and random.random() < self.epsilon:
            return self.env.action_space.sample()
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_tensor)
            return q_values.max(1)[1].item()
    
    def train_step(self):
        """Perform a single training step"""
        if len(self.replay_buffer) < self.batch_size:
            return
        
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.BoolTensor(dones).to(device)
        
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(1)[0]
            target_q_values = rewards + (self.gamma * next_q_values * ~dones)
        
        loss = F.mse_loss(current_q_values, target_q_values)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        
        if self.epsilon > self.epsilon_end:
            self.epsilon *= self.epsilon_decay
        
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.losses.append(loss.item())
    
    def train(self, num_episodes=500, max_steps=10000):
        """Train the agent"""
        print("Starting training...")
        
        for episode in range(num_episodes):
            obs, _ = self.env.reset()
            frame_stack = deque([self.preprocess_state(obs) for _ in range(self.input_channels)], maxlen=self.input_channels)
            state = self.stack_frames(frame_stack)
            
            episode_reward = 0
            episode_steps = 0
            
            for step in range(max_steps):
                action = self.select_action(state)
                
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated
                
                next_frame = self.preprocess_state(next_obs)
                frame_stack.append(next_frame)
                next_state = self.stack_frames(frame_stack)
                
                reward = np.clip(reward, -1, 1)
                
                self.replay_buffer.push(state, action, reward, next_state, done)

                self.train_step()
                
                state = next_state
                episode_reward += reward
                episode_steps += 1
                
                if done:
                    break
            
            self.episode_rewards.append(episode_reward)
            
            if (episode + 1) % 10 == 0:
                avg_reward = np.mean(self.episode_rewards[-10:])
                print(f"Episode {episode + 1}/{num_episodes}, "
                      f"Reward: {episode_reward:.2f}, "
                      f"Avg Reward: {avg_reward:.2f}, "
                      f"Epsilon: {self.epsilon:.3f}")
        
        print("Training completed!")
        
    def play(self, num_episodes=5, render=True):
        """Play the game using the trained agent"""
        print("Playing...")
        
        for episode in range(num_episodes):
            obs, _ = self.env.reset()
            frame_stack = deque([self.preprocess_state(obs) for _ in range(self.input_channels)], maxlen=self.input_channels)
            state = self.stack_frames(frame_stack)
            
            episode_reward = 0
            done = False
            
            while not done:
                if render:
                    img = self.env.render()
                    if isinstance(img, tuple):
                        img = img[0]
                    plt.imshow(img)
                    plt.axis('off')
                    plt.pause(0.01)
                
                action = self.select_action(state, training=False)
                obs, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated
                
                next_frame = self.preprocess_state(obs)
                frame_stack.append(next_frame)
                state = self.stack_frames(frame_stack)
                
                episode_reward += reward
                
                if done:
                    break
            
            print(f"Episode {episode + 1}: Reward = {episode_reward:.2f}")
        
        self.env.close()
    
    def save_model(self, path):
        """Save the model"""
        torch.save(self.q_network.state_dict(), path)
        print(f"Model saved to {path}")
    
    def load_model(self, path):
        """Load the model"""
        self.q_network.load_state_dict(torch.load(path, map_location=device))
        self.target_network.load_state_dict(self.q_network.state_dict())
        print(f"Model loaded from {path}")

In [12]:
agent = DQNAgent(
    env_name="Breakout-v4",
    lr=0.0001,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    buffer_capacity=100000,
    batch_size=32,
    target_update_freq=1000
)

agent.train(num_episodes=500)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(agent.episode_rewards)
plt.title('Episode Rewards')
plt.xlabel('Episode')
plt.ylabel('Reward')

plt.subplot(1, 2, 2)
plt.plot(agent.losses)
plt.title('Training Loss')
plt.xlabel('Training Step')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

agent.play(num_episodes=5)

agent.save_model("dqn_breakout.pth")

NameNotFound: Environment `Breakout` doesn't exist.